In [1]:
import os, openml, tqdm, json

In [2]:
import numpy as np
import pandas as pd

In [3]:
from pathlib import Path
from openml.tasks import TaskType
from sklearn.preprocessing import LabelEncoder

In [4]:
benchmark = openml.study.get_suite('tabarena-v0.1')
tasks = openml.tasks.list_tasks(task_id=benchmark.tasks, output_format='dataframe')
tasks = tasks[tasks['ttid'] == TaskType.SUPERVISED_CLASSIFICATION]
tasks = tasks['tid'].astype(int).tolist()
tasks[:5]

[363613, 363614, 363616, 363618, 363619]

In [5]:
def load_tabarena(tid):
    task = openml.tasks.get_task(tid)
    X, y, _, _ = task.get_dataset().get_data(target=task.target_name)
    return X, LabelEncoder().fit_transform(y)

In [6]:
for tid in tqdm.tqdm(tasks):
    X, y = load_tabarena(tid)
    data = np.column_stack([X, y])
    pd.DataFrame(data).to_csv(f'tabarena/{tid}.csv', index=False)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 38/38 [00:14<00:00,  2.66it/s]


In [7]:
def load_split(tid, split):
    blocks = []
    for prefix in ['N', 'C']:
        path = f'temp/{tid}/{prefix}_{split}.npy'
        if os.path.exists(path):
            blocks.append(np.load(path, allow_pickle=True))
    X = np.concatenate(blocks, axis=1)
    y = np.load(f'temp/{tid}/y_{split}.npy', allow_pickle=True)
    return X, y

In [8]:
def load_talent(tid):
    
    Xs, ys = [], []
    for split in ['train', 'test', 'val']:
        try:
            X, y = load_split(tid, split)
            Xs.append(X)
            ys.append(y)
        except:
            continue
     
    X = np.concatenate(Xs, axis=0)
    y = np.concatenate(ys, axis=0)
    return X, LabelEncoder().fit_transform(y)

In [9]:
tasks = os.listdir('temp/')
tasks[:5]

['Performance-Prediction',
 'Contaminant-detection-in-packaged-cocoa-hazelnut-spread-jars-using-Microwaves-Sensing-and-Machine-Learning-9.5GHz(Urbinati)',
 'philippine',
 'Fitness_Club_c',
 'dry_bean_dataset']

In [10]:
for tid in tqdm.tqdm(tasks):
    info = json.loads(Path(f'temp/{tid}/info.json').read_text())
    if info['task_type'] in ['binclass', 'multiclass']:
        X, y = load_talent(tid)
        data = np.column_stack([X, y])
        pd.DataFrame(data).to_csv(f'talent/{tid}.csv', index=False)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:16<00:00, 18.28it/s]


In [11]:
len(os.listdir('tabarena')), len(os.listdir('talent'))

(38, 200)